# Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import os
import sys
import pandas as pd

# enforce more deterministic behavior in cuBLAS operations.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# select a GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

sys.path.append("..")

from processor.core.interaction_conductor.llm_conductor import LLMConductor
from processor.core.ir_system.lm_interface import LMInterface
from processor.core.ir_system.ir_data_model import RetrieverType
from processor.utils.logger import setup_logger
from processor.core.ir_system.ir_data_model import AbstractDocument
from processor.core.ir_system.ir_data_model import Table, TableContext
from processor.core.ir_system.ir_data_model import Knowledge

In [3]:
# llm_path = "model/weight/qwen3-1_7b"
llm_path = "gpt-4o-mini"
embed_model_path = "model/weight/bge-base"
logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=logging.INFO,
    max_bytes=10_000_000,
    backup_count=5,
)
llm_conductor = LLMConductor(llm_path, embed_model_path, logger)

[2025-07-20 14:58:40] INFO in llm_conductor: Initializing LLMConductor with llm_path=gpt-4o-mini, embed_model_path=model/weight/bge-base
[2025-07-20 14:58:40] INFO in llm_planner: Initializing LLMPlanner


# IR Indexing

In [ ]:
DO_INDEXING_PNEUMA = False
DO_INDEXING_KB = False

In [ ]:
ir_system = LMInterface({
    "llm": llm_conductor.llm,
    "embed_model": llm_conductor.embed_model
})

In [ ]:
# Pneuma indexing
if DO_INDEXING_PNEUMA:
    TABLES_PATH = "../../data_src/buysite"
    DATASET_NAME = "buysite"
    documents: list[AbstractDocument] = []
    metadata = pd.read_csv(f"{TABLES_PATH}/metadata.csv")
    for table_fname in os.listdir(f"{TABLES_PATH}/dataset"):
        try:
            table_name = f"{TABLES_PATH}/dataset/{table_fname}"
            table = pd.read_csv(table_name, nrows=100)
            documents.append(Table(
                doc_id=table_name,
                retriever_type=RetrieverType.PNEUMA,
                content=table,
                metadata={
                    "table_name": table_name,
                    "dataset_name": DATASET_NAME,
                }
            ))
            table_context = metadata[metadata["table"].str.endswith(table_fname[:-4].upper())].reset_index(drop=True)
            if len(table_context) > 0:
                documents.append(
                    TableContext(
                        doc_id=f"table_context_{table_name}",
                        retriever_type=RetrieverType.PNEUMA,
                        content=table_context["value"][0],
                        metadata={
                            "table_name": table_name,
                            "dataset_name": DATASET_NAME,
                            "type": "description",
                        }
                    )
                )
        except pd.errors.EmptyDataError:
            continue
    ir_system.index_documents(
        RetrieverType.PNEUMA,
        documents,
    )

In [ ]:
# Test Pneuma's retrieval
# x = ir_system.retrieve_documents(
#     "Which item is bought by vendor A in our dataset?", ["buysite"], 1
# )
# x[RetrieverType.PNEUMA][0].doc_id

In [ ]:
# KB indexing
if DO_INDEXING_KB:
    documents: list[AbstractDocument] = [
        Knowledge(
            doc_id="kb_1",
            retriever_type=RetrieverType.KNOWLEDGE_BASE,
            content="",
            metadata={
                "type": "local",
                "user": "James"
            }
        )
    ]
    ir_system.index_documents(
        RetrieverType.KNOWLEDGE_BASE, documents
    )

In [ ]:
# Test KB's retrieval
# x = ir_system.retrieve_documents(
#     "Does user wants local or global information?", [], 1
# )
# x[RetrieverType.KNOWLEDGE_BASE][0]

# E2E Evaluation

## Basic Evaluation

In [4]:
QUESTION_1 = "I need to know if my team was preparing to receive shipping items on July 25, 2024."

In [5]:
llm_conductor.process_input(QUESTION_1)

[2025-07-20 14:58:50] INFO in llm_conductor: Start processing this user input: I need to know if my team was preparing to receive shipping items on July 25, 2024.
[2025-07-20 14:58:50] INFO in llm_conductor: Below is the current chat history:
[2025-07-20 14:58:50] INFO in llm_conductor: Conversation history:
User: I need to know if my team was preparing to receive shipping items on July 25, 2024.
[2025-07-20 14:58:52] INFO in llm_conductor: Model output: {
  "is_direct_response": false,
  "tool": "IR System",
  "response": {
    "prompt": "Retrieve documents regarding team preparations for receiving shipping items on July 25, 2024."
  }
}
[2025-07-20 14:58:52] INFO in llm_conductor: Parsed JSON output: {'is_direct_response': False, 'tool': 'IR System', 'response': {'prompt': 'Retrieve documents regarding team preparations for receiving shipping items on July 25, 2024.'}}
[2025-07-20 14:58:52] INFO in llm_conductor: Using tool: IR System
[2025-07-20 14:58:52] INFO in llm_conductor: IR S

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Initial retrieval returned 5 documents
Starting sanity check iteration 1
Found 4 irrelevant documents
Re-retrieving with feedback: The documents contain information that is not directly relevant to team preparations for receiving shipping items on July 25, 2024. Specifically, they reference past shipments and orders with dates prior to the requested date or do not provide any preparation-related context....


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Final results for RetrieverType.PNEUMA: 5 documents

Processing retriever: RetrieverType.KNOWLEDGE_BASE
Both the local and global retrievers have not been initialized.
Initial retrieval returned 0 documents
Final results for RetrieverType.KNOWLEDGE_BASE: 0 documents

Processing retriever: RetrieverType.WEB_SEARCH
Initial retrieval returned 0 documents
Final results for RetrieverType.WEB_SEARCH: 0 documents
Document retrieval completed
[2025-07-20 14:59:08] INFO in llm_conductor: Retrieved context length: 501
[2025-07-20 14:59:08] INFO in llm_conductor: Below is the current chat history:
[2025-07-20 14:59:08] INFO in llm_conductor: Conversation history:
User: I need to know if my team was preparing to receive shipping items on July 25, 2024.
[2025-07-20 14:59:09] INFO in llm_conductor: Model output: {
  "is_direct_response": false,
  "tool": "State Manipulation",
  "response": {
    "new_target_schemas": {
      "S1": ["ORG_ID", "PO_ID", "SHIPMENT_DATE", "PO_RECEIPT_STATUS"]
    },
    

ValueError: malformed node or string on line 1: <ast.Name object at 0x7fb2184e4a10>